# Heart Disease Prediction — Step 10: Save the Final Models

This notebook prepares everything the web app will need: all 4 trained models (so the app can show a
live comparison), the scaler, and the exact feature order — bundled cleanly so nothing has to be
retrained.

**Important distinction for the app and your report:**
- **XGBoost (tuned)** is the *deployed, primary* model — the one actually used for the official
  prediction. It earned that spot through the full training → tuning → evaluation process.
- The other 3 (Logistic Regression, Decision Tree, Random Forest) are saved too, purely so the app can
  show *"here's how the models we evaluated compare on this same patient"* as supporting evidence —
  not as competing final answers.

In [1]:
%pip install xgboost shap optuna joblib scikit-learn 

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import joblib
import json

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

RANDOM_STATE = 42

X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')
X_train_scaled = pd.read_csv('X_train_scaled.csv')
X_test_scaled = pd.read_csv('X_test_scaled.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
y_test = pd.read_csv('y_test.csv').squeeze()

scaler = joblib.load('scaler.joblib')

## 1. Load the already-tuned models

Random Forest and XGBoost were already tuned and saved back in the hyperparameter tuning step — we
just load them here rather than retraining.

In [3]:
best_rf = joblib.load('random_forest_tuned.joblib')
best_xgb = joblib.load('xgboost_tuned.joblib')

print("Loaded: Random Forest (tuned), XGBoost (tuned)")

Loaded: Random Forest (tuned), XGBoost (tuned)


## 2. Train and save Logistic Regression and Decision Tree

These two were trained during model comparison but never saved to a file — we do that now, using the
same settings as before, so the comparison in the app is consistent with what we evaluated.

In [4]:
log_reg = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)   # Logistic Regression uses scaled data, as before

dec_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
dec_tree.fit(X_train, y_train)         # Tree-based models use unscaled data

print("Trained: Logistic Regression, Decision Tree")

Trained: Logistic Regression, Decision Tree


## 3. Final sanity check — confirm all 4 models still perform as expected

A quick re-check against the test set, so we know we're saving models that actually match the results
already documented in the evaluation notebook.

In [5]:
def evaluate(name, model, Xte, needs_scaled=False):
    X_used = Xte
    y_pred = model.predict(X_used)
    y_proba = model.predict_proba(X_used)[:, 1]
    return {
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 3),
        'Precision': round(precision_score(y_test, y_pred), 3),
        'Recall': round(recall_score(y_test, y_pred), 3),
        'F1-score': round(f1_score(y_test, y_pred), 3),
        'AUC-ROC': round(roc_auc_score(y_test, y_proba), 3),
    }

summary = [
    evaluate('Logistic Regression', log_reg, X_test_scaled),
    evaluate('Decision Tree', dec_tree, X_test),
    evaluate('Random Forest (tuned)', best_rf, X_test),
    evaluate('XGBoost (tuned)', best_xgb, X_test),
]

final_summary_df = pd.DataFrame(summary).set_index('Model')
final_summary_df

,Accuracy,Precision,Recall,F1-score,AUC-ROC
Model,,,,,
Logistic Regression,0.715,0.700,0.620,0.658,0.743
Decision Tree,0.754,0.727,0.709,0.718,0.749
Random Forest (tuned),0.732,0.738,0.608,0.667,0.845
XGBoost (tuned),0.715,0.712,0.595,0.648,0.827


Matches what we already found in the Final Evaluation notebook — confirmed consistent.

## 4. Save all 4 models, the scaler, and the feature order

Everything the web app needs, in one place. We also save the exact column order and which columns
need scaling, so the app formats new patient input exactly the way the models expect.

In [6]:
joblib.dump(log_reg, 'model_logistic_regression.joblib')
joblib.dump(dec_tree, 'model_decision_tree.joblib')
joblib.dump(best_rf, 'model_random_forest.joblib')
joblib.dump(best_xgb, 'model_xgboost.joblib')   # this is the primary/deployed model
joblib.dump(scaler, 'scaler.joblib')

metadata = {
    'primary_model': 'XGBoost',
    'primary_model_file': 'model_xgboost.joblib',
    'comparison_models': {
        'Logistic Regression': 'model_logistic_regression.joblib',
        'Decision Tree': 'model_decision_tree.joblib',
        'Random Forest': 'model_random_forest.joblib',
    },
    'feature_order': list(X_train.columns),
    'scaled_columns': ['age', 'trestbps', 'chol', 'thalachh', 'oldpeak'],
    'scaler_file': 'scaler.joblib',
}

with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved:")
print("- model_xgboost.joblib          (primary/deployed model)")
print("- model_random_forest.joblib    (for comparison)")
print("- model_decision_tree.joblib    (for comparison)")
print("- model_logistic_regression.joblib (for comparison)")
print("- scaler.joblib")
print("- model_metadata.json           (feature order, which model is primary, etc.)")

Saved:
- model_xgboost.joblib          (primary/deployed model)
- model_random_forest.joblib    (for comparison)
- model_decision_tree.joblib    (for comparison)
- model_logistic_regression.joblib (for comparison)
- scaler.joblib
- model_metadata.json           (feature order, which model is primary, etc.)


## Summary

- All 4 evaluated models are saved and ready to load directly — no retraining needed in the web app
- **XGBoost is explicitly marked as the primary/deployed model** in `model_metadata.json`, so the app
  logic (and anyone reading the code) can clearly see it's the intended main prediction, not just one
  of four equal options
- The feature order and which columns need scaling are saved alongside the models, so new patient
  input can be formatted correctly without guessing

Next step: **build the web app**, once we've discussed how you want it to look.